In [ ]:
"""
Manhattan Plot: Full-scale GLM + BH FDR
========================================
X-axis: 4 outcomes x 5 categories (with gaps between outcomes)
Y-axis: -log10(BH FDR), truncated above threshold with sqrt compression
Diamond markers for features significant in all 4 outcomes.
"""

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests


# ========================= Configuration =========================
DATA_PATH = "./data/work/0219fourbrain_behavior_cov.csv"
MATCH_PATH = "./data/work/column_match_result.csv"
SAVE_DIR = "./data/work/BrainAging0218"
FDR_ALPHA = 0.05

OUTCOME_COLS = [
    "brain_difference",
    "baizhi_difference",
    "huizhi_difference",
    "protein_difference",
]
OUTCOME_LABELS = {
    "brain_difference": "Brain",
    "baizhi_difference": "White Matter",
    "huizhi_difference": "Gray Matter",
    "protein_difference": "Protein",
}
GLM_COVARIATES = ["age", "sex", "race", "APOE4_carrier"]

MANUAL_CATEGORY = {
    "Glycated.haemoglobin..HbA1c": "Biochemical markers",
    "summed_MET": "Lifestyles",
    "Average_total_household_income_before_tax": "Psychosocial factors",
}
CATEGORY_ORDER = [
    "Biochemical markers",
    "Body measure",
    "Lifestyles",
    "Food Intake",
    "Psychosocial factors",
]
CATEGORY_COLORS = {
    "Biochemical markers": "#E74C3C",
    "Body measure": "#F39C12",
    "Lifestyles": "#2ECC71",
    "Food Intake": "#3498DB",
    "Psychosocial factors": "#9B59B6",
}
COVARIATES_SET = {"eid", "age", "sex", "race", "bmi", "smoke", "drink", "APOE4_carrier"}

# Y-axis truncation threshold: values above this are sqrt-compressed
Y_TRUNCATE = 30


# ========================= GLM + FDR =========================

def glm_single(x_outcome, y_feature, cov_matrix):
    """Run a single OLS regression of Z-scored feature on outcome + covariates.
    Returns (beta, p-value).
    """
    mask = ~np.isnan(x_outcome) & ~np.isnan(y_feature)
    if cov_matrix is not None:
        mask &= ~np.any(np.isnan(cov_matrix), axis=1)
    n = mask.sum()
    min_n = 10 if cov_matrix is None else (cov_matrix.shape[1] + 5)
    if n < min_n:
        return np.nan, np.nan
    x, y = x_outcome[mask], y_feature[mask]
    sd = y.std()
    if sd == 0:
        return 0.0, np.nan
    y_z = (y - y.mean()) / sd
    try:
        if cov_matrix is not None:
            X = np.column_stack([np.ones(n), x, cov_matrix[mask]])
        else:
            X = np.column_stack([np.ones(n), x])
        model = sm.OLS(y_z, X).fit()
        return model.params[1], model.pvalues[1]
    except Exception:
        return np.nan, np.nan


def run_glm_bh(df, outcome_cols, features, glm_covariates):
    """Run GLM for all outcome x feature pairs and apply BH FDR correction.
    Returns a DataFrame with columns: outcome, feature, beta, p_raw, p_fdr, sig.
    """
    cov_cols = [c for c in glm_covariates if c in df.columns]
    cov_matrix = df[cov_cols].values.astype(np.float64) if cov_cols else None

    outcome_arr = {
        oc: pd.to_numeric(df[oc], errors="coerce").values.astype(np.float64)
        for oc in outcome_cols
    }
    feature_arr = {
        f: pd.to_numeric(df[f], errors="coerce").values.astype(np.float64)
        for f in features if f in df.columns
    }

    records = []
    for oc in outcome_cols:
        print(f"  GLM: {oc} ...")
        for f in features:
            if f not in feature_arr or f == oc or f in cov_cols:
                continue
            beta, pval = glm_single(outcome_arr[oc], feature_arr[f], cov_matrix)
            records.append({"outcome": oc, "feature": f, "beta": beta, "p_raw": pval})

    df_res = pd.DataFrame(records)
    df_res["p_fdr"] = np.nan
    df_res["sig"] = False

    # Per-outcome BH FDR correction
    for oc in outcome_cols:
        mask_oc = df_res["outcome"] == oc
        pvals = df_res.loc[mask_oc, "p_raw"].values
        pvals_clean = np.where(np.isnan(pvals), 1.0, pvals)
        reject, fdr_pvals, _, _ = multipletests(pvals_clean, alpha=FDR_ALPHA, method="fdr_bh")
        df_res.loc[mask_oc, "p_fdr"] = fdr_pvals
        df_res.loc[mask_oc, "sig"] = reject
        print(f"    {OUTCOME_LABELS.get(oc, oc)}: {reject.sum()}/{mask_oc.sum()} significant")

    return df_res


# ========================= Helpers =========================

def build_feature_category_map(match_path, manual_overrides):
    """Build a feature -> category mapping from the match file."""
    match_df = pd.read_csv(match_path)
    matched = match_df[match_df["CSV_col"].notna() & (match_df["CSV_col"] != "")]
    mapping = {}
    for _, row in matched.iterrows():
        csv_col = row["CSV_col"]
        cat = row["Category"]
        if pd.notna(cat) and cat != "":
            mapping[csv_col] = cat
    mapping.update(manual_overrides)
    return mapping


def select_features(df, feature_to_category, exclude_cols, min_obs=100):
    """Select valid feature columns with enough observations."""
    features = []
    for f in feature_to_category:
        if f in df.columns and f not in exclude_cols:
            df[f] = pd.to_numeric(df[f], errors="coerce")
            if df[f].notna().sum() >= min_obs:
                features.append(f)
    return features


def y_transform(val, y_truncate):
    """Y-axis truncation transform:
    - val <= y_truncate: identity
    - val >  y_truncate: compressed via sqrt
    """
    if val <= y_truncate:
        return val
    return y_truncate + np.sqrt(val - y_truncate) * 2


def find_consistent_features(glm_df, features):
    """Find features that are FDR-significant in all 4 outcomes."""
    consistent = set()
    for f in features:
        f_sub = glm_df[glm_df["feature"] == f]
        if len(f_sub) == len(OUTCOME_COLS) and f_sub["sig"].all():
            consistent.add(f)
    return consistent


# ========================= Plotting =========================

def build_plot_data(glm_df, features, feature_to_category, consistent_set):
    """Assign x-positions and compute display y-values for each point.
    Returns (plot_df, outcome_cat_centers, outcome_centers).
    """
    np.random.seed(42)
    records = []
    outcome_cat_centers = {}
    outcome_centers = {}
    x_offset = 0
    OUTCOME_GAP = 3.0
    CAT_GAP = 0.5

    for oc_idx, oc in enumerate(OUTCOME_COLS):
        if oc_idx > 0:
            x_offset += OUTCOME_GAP
        oc_x_start = x_offset

        for cat_idx, cat in enumerate(CATEGORY_ORDER):
            cat_features = [f for f in features if feature_to_category.get(f, "") == cat]
            if not cat_features:
                continue
            if cat_idx > 0:
                x_offset += CAT_GAP

            x_start = x_offset
            for i, f in enumerate(cat_features):
                row = glm_df[(glm_df["outcome"] == oc) & (glm_df["feature"] == f)]
                if row.empty:
                    continue
                fdr_p = row["p_fdr"].values[0]
                beta = row["beta"].values[0]
                is_sig = row["sig"].values[0]

                neg_log_raw = -np.log10(max(fdr_p, 1e-300))
                neg_log_display = y_transform(neg_log_raw, Y_TRUNCATE)
                x_pos = x_start + i * 0.18 + np.random.uniform(-0.04, 0.04)

                records.append({
                    "feature": f,
                    "category": cat,
                    "outcome": oc,
                    "outcome_label": OUTCOME_LABELS.get(oc, oc),
                    "x": x_pos,
                    "neg_log_fdr_raw": neg_log_raw,
                    "neg_log_fdr": neg_log_display,
                    "is_sig": is_sig,
                    "is_consistent": f in consistent_set,
                    "is_truncated": neg_log_raw > Y_TRUNCATE,
                    "beta": beta,
                })

            cat_width = max(len(cat_features) * 0.18, 1.0)
            outcome_cat_centers[(oc, cat)] = x_start + cat_width / 2
            x_offset = x_start + cat_width

        outcome_centers[oc] = (oc_x_start + x_offset) / 2

    return pd.DataFrame(records), outcome_cat_centers, outcome_centers


def plot_manhattan(plot_df, outcome_cat_centers, outcome_centers, save_path):
    """Draw the Manhattan plot and save to file."""
    if plot_df.empty:
        print("  No data to plot")
        return

    sig_line = -np.log10(FDR_ALPHA)
    sig_display = y_transform(sig_line, Y_TRUNCATE)
    y_trunc_display = y_transform(Y_TRUNCATE, Y_TRUNCATE)
    y_max_display = plot_df["neg_log_fdr"].max() * 1.08

    fig, ax = plt.subplots(figsize=(28, 9))

    # --- Background shading per category ---
    for oc in OUTCOME_COLS:
        sub = plot_df[plot_df["outcome"] == oc]
        for cat in CATEGORY_ORDER:
            cat_sub = sub[sub["category"] == cat]
            if cat_sub.empty:
                continue
            x_min = cat_sub["x"].min() - 0.2
            x_max = cat_sub["x"].max() + 0.2
            color = CATEGORY_COLORS.get(cat, "#CCCCCC")
            ax.axvspan(x_min, x_max, alpha=0.06, color=color, zorder=0)

    # --- Vertical separators between outcomes ---
    for i in range(len(OUTCOME_COLS) - 1):
        sub1 = plot_df[plot_df["outcome"] == OUTCOME_COLS[i]]
        sub2 = plot_df[plot_df["outcome"] == OUTCOME_COLS[i + 1]]
        if not sub1.empty and not sub2.empty:
            gap_x = (sub1["x"].max() + sub2["x"].min()) / 2
            ax.axvline(x=gap_x, color="#CCCCCC", linewidth=1.5,
                       linestyle="-", alpha=0.4, zorder=1)

    # --- Truncation zone indicator ---
    ax.axhline(y=y_trunc_display, color="#AAAAAA", linewidth=0.8,
               linestyle=":", alpha=0.5, zorder=1)
    ax.axhspan(y_trunc_display, y_max_display, alpha=0.04, color="grey", zorder=0)
    ax.text(plot_df["x"].max() + 0.8, y_trunc_display + 0.3,
            f"sqrt scale above {Y_TRUNCATE}",
            fontsize=7, color="#888888", va="bottom", ha="left", fontstyle="italic")

    # --- Scatter points ---
    for oc in OUTCOME_COLS:
        sub = plot_df[plot_df["outcome"] == oc]
        for cat in CATEGORY_ORDER:
            cat_sub = sub[sub["category"] == cat]
            if cat_sub.empty:
                continue
            color = CATEGORY_COLORS.get(cat, "#CCCCCC")

            # Non-significant
            ns = cat_sub[~cat_sub["is_sig"]]
            if not ns.empty:
                ax.scatter(ns["x"], ns["neg_log_fdr"],
                           c=color, s=15, alpha=0.3, edgecolors="none", zorder=2)

            # Significant but not consistent
            sig_only = cat_sub[cat_sub["is_sig"] & ~cat_sub["is_consistent"]]
            if not sig_only.empty:
                ax.scatter(sig_only["x"], sig_only["neg_log_fdr"],
                           c=color, s=35, alpha=0.7, edgecolors="white",
                           linewidth=0.3, zorder=3)

            # Consistent (significant in all 4 outcomes)
            consist = cat_sub[cat_sub["is_consistent"]]
            if not consist.empty:
                ax.scatter(consist["x"], consist["neg_log_fdr"],
                           c=color, s=90, alpha=1.0, edgecolors="black",
                           linewidth=1.2, zorder=4, marker="D")
                # Triangle marker for truncated points
                trunc = consist[consist["is_truncated"]]
                if not trunc.empty:
                    ax.scatter(trunc["x"], trunc["neg_log_fdr"] + 0.3,
                               marker="^", c="black", s=20, zorder=5)

    # --- FDR significance line ---
    ax.axhline(y=sig_display, color="#E74C3C", linewidth=1.2,
               linestyle="--", alpha=0.7, zorder=1)
    ax.text(plot_df["x"].max() + 0.5, sig_display,
            f"FDR = {FDR_ALPHA}", fontsize=9, color="#E74C3C",
            va="center", ha="left", fontweight="bold")

    # --- Label top consistent features per outcome ---
    TOP_N_LABEL = 10
    for oc in OUTCOME_COLS:
        label_sub = plot_df[
            (plot_df["outcome"] == oc) &
            plot_df["is_consistent"] &
            plot_df["is_sig"]
        ]
        if label_sub.empty:
            continue
        label_sub = label_sub.sort_values("neg_log_fdr_raw", ascending=False)
        to_label = label_sub.head(TOP_N_LABEL)

        y_range = plot_df["neg_log_fdr"].max()
        min_y_gap = y_range * 0.035
        used_y = []

        for _, row in to_label.iterrows():
            y_pos = row["neg_log_fdr"]
            for uy in used_y:
                if abs(y_pos - uy) < min_y_gap:
                    y_pos = uy + min_y_gap
            used_y.append(y_pos)

            # Show raw value in label for truncated points
            if row["is_truncated"]:
                label_text = f"{row['feature']} ({row['neg_log_fdr_raw']:.0f})"
            else:
                label_text = row["feature"]

            ax.annotate(
                label_text,
                xy=(row["x"], row["neg_log_fdr"]),
                xytext=(row["x"] + 0.4, y_pos + y_range * 0.008),
                fontsize=5.5, fontweight="bold",
                color=CATEGORY_COLORS.get(row["category"], "black"),
                arrowprops=dict(arrowstyle="-", color="#AAAAAA", lw=0.4),
                ha="left", va="bottom", zorder=5,
            )

    # --- X-axis: category labels below each outcome ---
    for oc in OUTCOME_COLS:
        for cat in CATEGORY_ORDER:
            key = (oc, cat)
            if key in outcome_cat_centers:
                ax.text(outcome_cat_centers[key], -0.02,
                        cat.split()[0],  # Abbreviated: first word only
                        fontsize=6, fontweight="bold",
                        color=CATEGORY_COLORS.get(cat, "black"),
                        ha="center", va="top",
                        transform=ax.get_xaxis_transform(),
                        rotation=35)

    # --- X-axis: outcome labels ---
    for oc in OUTCOME_COLS:
        if oc in outcome_centers:
            ax.text(outcome_centers[oc], -0.09,
                    OUTCOME_LABELS.get(oc, oc),
                    fontsize=13, fontweight="bold", color="#2C3E50",
                    ha="center", va="top",
                    transform=ax.get_xaxis_transform())

    # --- Y-axis: mixed linear + compressed ticks ---
    linear_ticks = list(range(0, Y_TRUNCATE + 1, 5))
    raw_ticks_above = [50, 80, 100, 150]
    above_ticks = [(t, y_transform(t, Y_TRUNCATE)) for t in raw_ticks_above
                   if y_transform(t, Y_TRUNCATE) <= y_max_display]

    all_tick_pos = linear_ticks + [t[1] for t in above_ticks]
    all_tick_lbl = [str(t) for t in linear_ticks] + [str(t[0]) for t in above_ticks]
    ax.set_yticks(all_tick_pos)
    ax.set_yticklabels(all_tick_lbl)

    ax.set_xticks([])
    ax.set_ylabel("$-\\log_{10}$(BH FDR)", fontsize=13, fontweight="bold")
    ax.set_ylim(-1, y_max_display)
    ax.set_xlim(plot_df["x"].min() - 1, plot_df["x"].max() + 3)

    ax.set_title(
        "Manhattan Plot: BH FDR Across 4 Brain Aging Outcomes\n"
        "(each outcome shows 5 categories; ◆ = FDR significant in all 4 outcomes)",
        fontsize=14, fontweight="bold", pad=15,
    )

    # --- Legend ---
    legend_elements = [
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor=CATEGORY_COLORS[cat],
               markersize=8, label=cat)
        for cat in CATEGORY_ORDER
    ]
    legend_elements.extend([
        Line2D([0], [0], marker="D", color="w", markerfacecolor="grey",
               markeredgecolor="black", markersize=9,
               label="Consistent (all 4 outcomes)"),
        Line2D([0], [0], linestyle="--", color="#E74C3C", linewidth=1.2,
               label=f"FDR = {FDR_ALPHA}"),
    ])
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8,
              framealpha=0.9, edgecolor="#CCCCCC")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    plt.subplots_adjust(bottom=0.16)
    plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
    print(f"  Saved: {save_path}")
    plt.close()


# ========================= Main =========================

def main():
    print("=" * 70)
    print("Manhattan Plot: BH FDR, 4 outcomes x 5 categories, Y-axis truncated")
    print("=" * 70)

    # ----- Data loading -----
    df = pd.read_csv(DATA_PATH)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    os.makedirs(SAVE_DIR, exist_ok=True)

    # ----- Feature-to-category mapping -----
    feature_to_category = build_feature_category_map(MATCH_PATH, MANUAL_CATEGORY)

    # ----- Feature selection -----
    exclude = set(OUTCOME_COLS) | COVARIATES_SET | {"avg_outcome"}
    features = select_features(df, feature_to_category, exclude)
    print(f"Features: {len(features)}")

    # ----- Remove bottom 50 extreme values per outcome -----
    for oc in OUTCOME_COLS:
        df[oc] = pd.to_numeric(df[oc], errors="coerce")
        valid = df[df[oc].notna()].sort_values(oc)
        drop_idx = valid.iloc[:50].index
        df = df.drop(index=drop_idx)
    print(f"After removing extremes: {len(df)} rows")

    # ----- GLM + BH FDR -----
    print("\n--- GLM + BH FDR ---")
    glm_df = run_glm_bh(df, OUTCOME_COLS, features, GLM_COVARIATES)

    # ----- Find consistent features (significant in all 4) -----
    consistent_set = find_consistent_features(glm_df, features)
    print(f"\nConsistent across all 4 outcomes: {len(consistent_set)}")

    # ----- Build plot data -----
    plot_df, outcome_cat_centers, outcome_centers = build_plot_data(
        glm_df, features, feature_to_category, consistent_set
    )

    # ----- Draw Manhattan plot -----
    save_path = os.path.join(SAVE_DIR, "Manhattan_4outcomes_BH_FDR.pdf")
    plot_manhattan(plot_df, outcome_cat_centers, outcome_centers, save_path)

    # ----- Summary statistics -----
    print("\n--- Summary ---")
    for oc in OUTCOME_COLS:
        sub = plot_df[plot_df["outcome"] == oc]
        n_sig = sub["is_sig"].sum()
        n_consist = sub["is_consistent"].sum()
        n_trunc = sub["is_truncated"].sum()
        label = OUTCOME_LABELS.get(oc, oc)
        print(f"  {label}: {n_sig} significant, {n_consist} consistent, {n_trunc} truncated")

    # ----- Save results table -----
    csv_path = os.path.join(SAVE_DIR, "Manhattan_4outcomes_data.csv")
    glm_df.to_csv(csv_path, index=False)
    print(f"  Results table: {csv_path}")


if __name__ == "__main__":
    main()